In [3]:
from pathlib import Path
import shutil
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.utils import Bunch

def load_cifar10(data_id: int = 40927):
    """优先从 OpenML 读取；若远端校验异常，则回退到 torchvision。"""
    try:
        return fetch_openml(data_id=data_id, as_frame=False)
    except ValueError as e:
        if "md5 checksum" not in str(e):
            raise

        # 清理缓存后再试一次
        cache_dir = Path.home() / "scikit_learn_data" / "openml"
        if cache_dir.exists():
            shutil.rmtree(cache_dir)
        try:
            return fetch_openml(data_id=data_id, as_frame=False)
        except ValueError as e2:
            if "md5 checksum" not in str(e2):
                raise

            # OpenML 仍失败时使用 torchvision 作为稳定回退方案
            from torchvision.datasets import CIFAR10

            train_ds = CIFAR10(root="./data", train=True, download=True)
            test_ds = CIFAR10(root="./data", train=False, download=True)

            X = np.concatenate([train_ds.data, test_ds.data], axis=0).reshape(-1, 32 * 32 * 3)
            y = np.array(train_ds.targets + test_ds.targets).astype(str)

            return Bunch(
                data=X,
                target=y,
                DESCR="CIFAR-10 loaded via torchvision fallback because OpenML checksum mismatched.",
                details={"source": "torchvision-fallback"},
            )

cifar10 = load_cifar10()
print("Loaded:", cifar10.data.shape, "labels:", len(cifar10.target))

100%|██████████| 170M/170M [06:50<00:00, 415kB/s]  
/home/shenss/python/DiffSo/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Loaded: (60000, 3072) labels: 60000


In [4]:
cifar10.DESCR

'CIFAR-10 loaded via torchvision fallback because OpenML checksum mismatched.'